<a href="https://colab.research.google.com/github/rafi-amiruddin/DVTE/blob/main/DVTE_Chapter_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Visualization Techniques for Economists

## Chapter 1

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Enables headless rendering for server/headless environments
import matplotlib.pyplot as plt
import matplotlib.dates
import matplotlib.ticker
import seaborn as sns
from matplotlib.patches import Patch
import matplotlib.ticker as ticker



# ==========================================
# 0. GLOBAL STYLE & THEME CONFIGURATION
# ==========================================
# ggplot2-inspired aesthetic for publication-quality visuals with serif fonts


sns.set_theme(style='darkgrid', font='sans-serif', font_scale=1, color_codes=True, rc=None)
# Output directory for the generated vector files
OUTPUT_DIR = "Figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [18]:
# ==========================================
# 1. CHART: SEMMELWEIS MONTHLY MORTALITY
# ==========================================
def generate_semmelweis_chart():
    """
    Semmelweis handwashing intervention: maternal mortality rates over time.
    Demonstrates the causal effect of chlorinated lime handwashing.
    """
    data = {
        "date": [
            "1846-01", "1846-02", "1846-03", "1846-04", "1846-05", "1846-06",
            "1846-07", "1846-08", "1846-09", "1846-10", "1846-11", "1846-12",
            "1847-01", "1847-02", "1847-03", "1847-04", "1847-05", "1847-06",
            "1847-07", "1847-08", "1847-09", "1847-10", "1847-11", "1847-12",
            "1848-01", "1848-02", "1848-03", "1848-04", "1848-05", "1848-06",
            "1848-07", "1848-08", "1848-09", "1848-10", "1848-11", "1848-12",
            "1849-01", "1849-02", "1849-03"
        ],
        "mortality_rate": [
            13.0, 15.0, 16.2, 18.5, 12.5, 10.1, 11.2, 18.0, 13.5, 14.2, 11.5, 10.0,
            11.7, 2.0, 3.5, 9.8, 12.2, 2.2, 1.2, 1.4, 1.8, 2.1, 1.5, 1.3,
            3.5, 1.1, 0.0, 0.5, 0.9, 1.1, 1.3, 0.0, 0.6, 1.2, 1.5, 1.8,
            2.3, 3.1, 4.9
        ]
    }

    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])

    # Separating pre-intervention and post-intervention eras
    # Handwashing mandated mid-May 1847
    intervention_date = pd.to_datetime("1847-05-15")
    df['era'] = np.where(df['date'] < intervention_date, 'Pre-Intervention', 'Post-Intervention')

    fig, ax = plt.subplots(figsize=(14, 6))

    # Plotting distinct color eras to emphasize the structural break
    pre_df = df[df['era'] == 'Pre-Intervention']
    post_df = df[df['era'] == 'Post-Intervention']

    # Plot lines for each era
    ax.plot(pre_df['date'], pre_df['mortality_rate'],
            linewidth=0.5, label='Before Antiseptic Handwashing',
            marker='o', markersize=5, alpha=0.85)
    ax.plot(post_df['date'], post_df['mortality_rate'],
            linewidth=0.5, label='After Chlorine Handwashing Mandate',
            marker='o', markersize=5, alpha=0.85)

    # Formatting X-Axis for time readability
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%b %Y'))
    plt.xticks(rotation=0)

    # Adding vertical intervention timeline marker
    ax.axvline(pd.to_datetime("1847-05-01"),
               linestyle='--', linewidth=2, alpha=0.6)
    ax.text(pd.to_datetime("1847-05-15"), 15.0,
            "May 1847:\nChlorinated Lime\nHandwashing Mandate",
            fontsize=10, fontweight='bold',
            bbox=dict(alpha=0.9, linewidth=0.5))

    # Title and Labels (Insight-driven title)
    ax.set_title("Chlorine Handwashing Mandate Caused Clinic Mortality to Plummet from 12.2% to 2.2% in One Month",
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel("Timeline (Monthly Intervals)", fontsize=11, labelpad=10)
    ax.set_ylabel("Maternal Mortality Rate (%)", fontsize=11, labelpad=10)
    ax.set_ylim(-0.5, 21.0)

    # ggplot2-style legend
    ax.legend(loc='upper right', frameon=True,
              framealpha=0.95, fontsize=10)

    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.6)
    ax.set_axisbelow(True)

    plt.tight_layout()

    # Save as Vector SVG
    svg_path = os.path.join(OUTPUT_DIR, "Fig_01_semmelweis.svg")
    fig.savefig(svg_path, format='svg', bbox_inches='tight', dpi=300)
    print(f"✓ Successfully generated: {svg_path}")
    plt.close()

generate_semmelweis_chart()

✓ Successfully generated: Figures/Fig_01_semmelweis.svg


In [19]:
# ==========================================
# 2. CHART: NCLB MATH PROFICIENCY
# ==========================================
"""
NCLB math proficiency: diverging stacked bars, 4th and 8th grade, 2000-2015.
Baseline sits at the Proficient / Basic cut, so the reader inverts the
mapping instantly: everything above the line is proficient, everything
below it is not.
"""
# Palette mirrors the source infographic: blues = proficient, pinks = not
C_ADVANCED = "#1B6FD1"
C_PROFICIENT = "#7EC8F2"
C_BASIC = "#F7A8C4"
C_BELOW = "#F0348B"


def generate_nclb_chart():
    """
    NAEP math achievement distributions, 2000-2015, by grade.
    Diverging stacked bars anchored at the proficiency threshold,
    with 2015 summary statistics on the right-hand margin.
    """
    # Values read directly off the NAEP achievement-level infographic
    data = {
        "Year": [2000, 2003, 2005, 2007, 2009, 2011, 2013, 2015] * 2,
        "Grade": ["4th-Grade Math"] * 8 + ["8th-Grade Math"] * 8,
        "Advanced":    [3, 4, 5, 6, 6, 7, 8, 7,  5, 5, 6, 7, 8, 8, 9, 8],
        "Proficient":  [21, 29, 31, 34, 33, 34, 34, 33,  21, 23, 24, 25, 26, 26, 27, 25],
        "Basic":       [42, 45, 44, 43, 43, 42, 41, 42,  38, 39, 39, 39, 39, 39, 38, 38],
        "Below Basic": [35, 23, 20, 18, 18, 18, 17, 18,  37, 32, 31, 29, 27, 27, 26, 29],
    }

    df = pd.DataFrame(data)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    grades = ["4th-Grade Math", "8th-Grade Math"]
    axes = [ax1, ax2]

    for grade, ax in zip(grades, axes):
        g = df[df["Grade"] == grade].reset_index(drop=True)
        x = range(len(g))
        width = 0.62

        # --- Above the line: proficient population ---
        ax.bar(x, g["Proficient"], width, color=C_PROFICIENT,
               edgecolor="white", linewidth=0.8, zorder=3)
        ax.bar(x, g["Advanced"], width, bottom=g["Proficient"],
               color=C_ADVANCED, edgecolor="white", linewidth=0.8, zorder=3)

        # --- Below the line: non-proficient population ---
        ax.bar(x, -g["Basic"], width, color=C_BASIC,
               edgecolor="white", linewidth=0.8, zorder=3)
        ax.bar(x, -g["Below Basic"], width, bottom=-g["Basic"],
               color=C_BELOW, edgecolor="white", linewidth=0.8, zorder=3)

        # --- Segment labels ---
        for i in x:
            ax.text(i, g["Proficient"][i] / 2, f"{g['Proficient'][i]}",
                    ha="center", va="center", fontsize=9.5, color="#0B3C6B", zorder=4)
            ax.text(i, g["Proficient"][i] + g["Advanced"][i] + 2.2,
                    f"{g['Advanced'][i]}", ha="center", va="center",
                    fontsize=9.5, color="#0B3C6B", zorder=4)
            ax.text(i, -g["Basic"][i] / 2, f"{g['Basic'][i]}",
                    ha="center", va="center", fontsize=9.5, color="#8B2252", zorder=4)
            ax.text(i, -g["Basic"][i] - g["Below Basic"][i] / 2,
                    f"{g['Below Basic'][i]}", ha="center", va="center",
                    fontsize=9.5, color="white", fontweight="bold", zorder=4)

        # --- Proficiency threshold ---
        ax.axhline(0, color="#333333", linewidth=1.6, zorder=5)

        # --- NCLB structural break (enacted Jan 2002, between 2000 and 2003) ---
        ax.axvline(0.5, color="#333333", linestyle="--", linewidth=1.3, zorder=2)
        if grade == "4th-Grade Math":
            ax.text(0.5, 45, "NCLB", ha="center", va="bottom",
                    fontsize=12, fontweight="bold", color="#222222")

        # --- Right-hand summary for the terminal (2015) year ---
        prof_2015 = g["Advanced"].iloc[-1] + g["Proficient"].iloc[-1]
        not_2015 = g["Basic"].iloc[-1] + g["Below Basic"].iloc[-1]

        ax.axhline(0, xmin=0.86, xmax=1.0, color="#999999",
                   linestyle="--", linewidth=1.1, zorder=1)
        ax.text(8.1, 26, f"{prof_2015}%", ha="left", va="center",
                fontsize=26, color="#222222")
        ax.text(8.1, 14, "Proficient", ha="left", va="center",
                fontsize=12, color=C_PROFICIENT, fontweight="bold")
        ax.text(8.1, -30, f"{not_2015}%", ha="left", va="center",
                fontsize=26, color="#222222")
        ax.text(8.1, -42, "Not Proficient", ha="left", va="center",
                fontsize=12, color=C_BELOW, fontweight="bold")

        ax.set_title(grade, fontsize=14, fontweight="bold", pad=14, loc="left")
        ax.set_xlim(-0.7, 9.6)
        ax.set_ylim(-84, 52)
        ax.set_xticks(list(x))
        ax.set_xticklabels(g["Year"])
        ax.set_yticks([])

        for side in ["top", "right", "left", "bottom"]:
            ax.spines[side].set_visible(False)
        ax.grid(False)

    ax1.tick_params(axis="x", length=0)
    ax2.tick_params(axis="x", length=0)
    ax2.set_xlabel("NAEP Assessment Year", fontsize=11, labelpad=10)
    ax1.set_ylabel("Share of Students (%)", fontsize=11, labelpad=10)
    ax2.set_ylabel("Share of Students (%)", fontsize=11, labelpad=10)

    handles = [
        Patch(facecolor=C_ADVANCED, label="Advanced"),
        Patch(facecolor=C_PROFICIENT, label="Proficient"),
        Patch(facecolor=C_BASIC, label="Basic"),
        Patch(facecolor=C_BELOW, label="Below Basic"),
    ]
    ax1.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 1.22),
               ncol=4, frameon=False, fontsize=10)

    fig.suptitle(
        "NCLB Visual Check: 60% of 4th Graders and 67% of 8th Graders Were Still "
        "Not Proficient in Math by 2015",
        fontsize=15, fontweight="bold", y=0.99)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    svg_path = os.path.join(OUTPUT_DIR, "Fig_01_nclb.svg")
    fig.savefig(svg_path, format="svg", bbox_inches="tight", dpi=300)
    png_path = os.path.join(OUTPUT_DIR, "Fig_01_nclb.png")
    fig.savefig(png_path, format="png", bbox_inches="tight", dpi=200)
    print(f"Successfully generated: {svg_path}")
    plt.close()


if __name__ == "__main__":
    generate_nclb_chart()

Successfully generated: Figures/Fig_01_nclb.svg


In [20]:
"""
Companion to the diverging stacked bar: the same NAEP data unstacked into
four independent tier-lines, faceted by grade on a shared vertical scale.
"""
C_ADVANCED = "#1B6FD1"
C_PROFICIENT = "#7EC8F2"
C_BASIC = "#F7A8C4"
C_BELOW = "#F0348B"

# ==========================================
# 2a. CHART: NCLB MATH PROFICIENCY (MULTI-LINE)
# ==========================================
def generate_nclb_line_chart():
    """
    NCLB math achievement trends, 2000-2015, unstacked.
    Four achievement tiers tracked as independent lines, faceted by grade
    on a shared vertical scale so each tier is judged by position rather
    than by segment length.
    """
    # NAEP achievement-level shares, transcribed from the source infographic
    data = {
        "Year": [2000, 2003, 2005, 2007, 2009, 2011, 2013, 2015] * 2,
        "Grade": ["4th-Grade Math"] * 8 + ["8th-Grade Math"] * 8,
        "Below Basic": [35, 23, 20, 18, 18, 18, 17, 18,
                        37, 32, 31, 29, 27, 27, 26, 29],
        "Basic":       [42, 45, 44, 43, 43, 42, 41, 42,
                        38, 39, 39, 39, 39, 39, 38, 38],
        "Proficient":  [21, 29, 31, 34, 33, 34, 34, 33,
                        21, 23, 24, 25, 26, 26, 27, 25],
        "Advanced":    [3, 4, 5, 6, 6, 7, 8, 7,
                        5, 5, 6, 7, 8, 8, 9, 8],
    }

    df = pd.DataFrame(data)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    grades = ["4th-Grade Math", "8th-Grade Math"]
    axes = [ax1, ax2]
    categories = ["Below Basic", "Basic", "Proficient", "Advanced"]
    colors = {"Below Basic": C_BELOW, "Basic": C_BASIC,
              "Proficient": C_PROFICIENT, "Advanced": C_ADVANCED}

    for grade, ax in zip(grades, axes):
        grade_df = df[df["Grade"] == grade]

        # Each tier is now its own line, read against the same vertical axis
        for cat in categories:
            ax.plot(grade_df["Year"], grade_df[cat], label=cat,
                    color=colors[cat], linewidth=2, marker="o",
                    markersize=6, alpha=0.9)

            # Label end-points (2015) directly, so no legend round-trip is needed
            final_val = grade_df[grade_df["Year"] == 2015][cat].values[0]
            ax.text(2015.4, final_val, f"{cat}  {final_val}%",
                    va="center", fontsize=9.5, color=colors[cat],
                    fontweight="bold")

        # NCLB enacted January 2002, between the 2000 and 2003 assessments
        ax.axvline(2001.5, color="#666666", linestyle="--", linewidth=1.2, zorder=1)
        ax.text(2001.5, 51, "NCLB", ha="center", va="bottom",
                fontsize=11, fontweight="bold", color="#333333")

        ax.set_title(f"{grade} Performance Distributions",
                     fontsize=13, fontweight="bold", pad=15)
        ax.set_xlabel("Assessment Year", fontsize=11, labelpad=10)
        ax.set_xlim(1999, 2022)
        ax.set_ylim(0, 55)
        ax.set_xticks([2000, 2003, 2005, 2007, 2009, 2011, 2013, 2015])

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(True, alpha=0.3, linestyle="-", linewidth=0.6)
        ax.set_axisbelow(True)

    ax1.set_ylabel("Percentage of Student Population (%)", fontsize=11, labelpad=10)
    # No legend: every tier is labeled directly at its 2015 end-point

    fig.suptitle(
        "Unstacked: Below Basic Fell Sharply Only Before 2007, While Basic "
        "Never Moved and Proficiency Plateaued Near One Third",
        fontsize=15, fontweight="bold", y=1.00)

    plt.tight_layout()

    svg_path = os.path.join(OUTPUT_DIR, "Fig_01_nclb_a.svg")
    fig.savefig(svg_path, format="svg", bbox_inches="tight", dpi=300)
    fig.savefig(os.path.join(OUTPUT_DIR, "Fig_01_nclb_a.png"),
                format="png", bbox_inches="tight", dpi=200)
    print(f"Successfully generated: {svg_path}")
    plt.close()


if __name__ == "__main__":
    generate_nclb_line_chart()

Successfully generated: Figures/Fig_01_nclb_a.svg


In [21]:
# ==========================================
# 3. CHART: EDUCATION SPENDING VS. PISA SCORES (SCATTER)
# ==========================================
def generate_pisa_chart():
    """
    PISA 2015 mathematics scores vs. education spending per student.
    Highlights the US as a high-spending, low-performance outlier.
    """
    # Realistic PISA 2015 data with OECD context
    data = {
        "Country": ["Estonia", "South Korea", "Japan", "Canada", "United States", "Luxembourg"],
        "Spending": [8080, 11200, 9597, 10626, 15841, 23456],
        "Score": [520, 524, 532, 516, 470, 486]
    }

    df = pd.DataFrame(data)

    # OECD benchmarks
    avg_spending = 10220
    avg_score = 490

    fig, ax = plt.subplots(figsize=(10, 8))

    # Constructing Scatter Plot with ggplot2 aesthetic
    ax.scatter(df["Score"], df["Spending"], s=250,
               edgecolors='white', linewidths=1.5, alpha=0.85, zorder=5)

    # Adding Quadrant lines (OECD Averages)
    ax.axvline(avg_score, linestyle='--',
               linewidth=1.5, alpha=0.5, zorder=1)
    ax.axhline(avg_spending, linestyle='--',
               linewidth=1.5, alpha=0.5, zorder=1)

    # Quadrant Label Annotations
    quadrant_kwargs = dict(fontsize=9, style='italic',
                          bbox=dict(facecolor='white', alpha=0.85,
                                   edgecolor='#e5e5e5', linewidth=0.5))
    ax.text(450, 21000, "High Spend / Low Score\n(Inefficient Input)", **quadrant_kwargs)
    ax.text(510, 21000, "High Spend / High Score\n(High-Resource Success)", **quadrant_kwargs)
    ax.text(450, 5000, "Low Spend / Low Score\n(Underfunded)", **quadrant_kwargs)
    ax.text(510, 5000, "Low Spend / High Score\n(High-Efficiency Yield)", **quadrant_kwargs)

    # Labeling Countries directly on the coordinate plane
    for idx, row in df.iterrows():
        fontweight = 'bold' if row['Country'] == 'United States' else 'normal'
        ax.text(row['Score'] + 1.5, row['Spending'], row['Country'],
                fontsize=10, fontweight=fontweight, va='center', zorder=6)

    # Title and Labels (Insight-focused)
    ax.set_title("The US Outlier: High Education Spending Fails to Yield Top-Tier Math Scores",
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel("PISA 2015 Math Score (Points)", fontsize=11, labelpad=10)
    ax.set_ylabel("Annual Spending per Student (USD)", fontsize=11, labelpad=10)

    # Format Y axis numbers with thousands separator
    ax.get_yaxis().set_major_formatter(
        matplotlib.ticker.FuncFormatter(lambda x, p: format(int(x), ','))
    )

    ax.set_xlim(440, 550)
    ax.set_ylim(4000, 24000)

    # ggplot2-style spines and grid
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.6)
    ax.set_axisbelow(True)

    plt.tight_layout()

    # Save as Vector SVG
    svg_path = os.path.join(OUTPUT_DIR, "Fig_01_pisa.svg")
    fig.savefig(svg_path, format='svg', bbox_inches='tight', dpi=300)
    print(f"✓ Successfully generated: {svg_path}")
    plt.close()
generate_pisa_chart()

✓ Successfully generated: Figures/Fig_01_pisa.svg


In [24]:

# ==========================================
# CHART 4: POTENTIALLY MISLEADING vs. NOT MISLEADING
# ==========================================
def generate_misleading_comparison():
    """
    Side-by-side bar charts comparing potentially misleading
    (truncated Y-axis) vs. not misleading (full scale) visualization.
    """

    # Same data for both charts
    categories = ["Category A", "Category B", "Category C", "Category D"]
    values = [77, 85, 91, 94]

    # Create figure with 2 subplots side-by-side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # ========== LEFT PLOT: POTENTIALLY MISLEADING ==========
    # Truncated Y-axis to exaggerate differences
    bars1 = ax1.bar(categories, values, color='#6b5b95', alpha=0.85,
                    edgecolor='white', linewidth=1.5, width=0.6)

    ax1.set_ylim(75, 95)  # Truncated axis — exaggerates differences
    ax1.set_ylabel("Score", fontsize=11)
    ax1.set_title("Potentially\nMisleading", fontsize=13, fontweight='bold', pad=15)

    # Add value labels on bars
    for bar, value in zip(bars1, values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.3,
                f'{value}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Styling
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_linewidth(0.8)
    ax1.spines['bottom'].set_linewidth(0.8)
    ax1.grid(True, axis='y', alpha=0.3, linestyle='-', linewidth=0.5)
    ax1.set_axisbelow(True)
    ax1.tick_params(axis='x', labelsize=10)
    ax1.tick_params(axis='y', labelsize=9)


    # ========== RIGHT PLOT: NOT MISLEADING ==========
    # Full Y-axis (0-100) — fair comparison
    bars2 = ax2.bar(categories, values, color='#6b5b95', alpha=0.85,
                    edgecolor='white', linewidth=1.5, width=0.6)

    ax2.set_ylim(0, 100)  # Full scale — honest representation
    ax2.set_ylabel("Score", fontsize=11)
    ax2.set_title("Not\nMisleading", fontsize=13, fontweight='bold', pad=15)

    # Add value labels on bars
    for bar, value in zip(bars2, values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1.5,
                f'{value}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Styling (same as left)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['left'].set_linewidth(0.8)
    ax2.spines['bottom'].set_linewidth(0.8)
    ax2.grid(True, axis='y', alpha=0.3, linestyle='-', linewidth=0.5)
    ax2.set_axisbelow(True)
    ax2.tick_params(axis='x', labelsize=10)
    ax2.tick_params(axis='y', labelsize=9)

    # Remove x-axis tick labels to match your image
    ax1.set_xticklabels([])
    ax2.set_xticklabels([])

    # Adjust layout
    plt.tight_layout()

    # Save as Vector SVG
    svg_path = os.path.join(OUTPUT_DIR, "Fig_01_misleading.svg")
    fig.savefig(svg_path, format='svg', bbox_inches='tight', dpi=300)
    print(f"✓ Successfully generated: {svg_path}")
    plt.close()


# Run it
generate_misleading_comparison()

✓ Successfully generated: Figures/Fig_01_misleading.svg


In [23]:
"""
Dual y-axis manipulation versus an honest common scale.

Panel A overlays two series on two axes whose ranges are chosen by
least-squares alignment. The scaling is fitted, not found: a modest
correlation is inflated into apparent lockstep by the right axis alone.

Panel B replots the identical numbers as percent change from a unified
2005 base on one shared scale. The co-movement disappears.
"""

# ==========================================
# CHART 5: Dual Y-Axis Manipulation vs. Honest Common Scale
# ==========================================

SEED = 131
C_INFL = "#C44E52"
C_UNEMP = "#4C72B0"


def build_data(seed=SEED):
    """Two independently drawn random walks: inflation and unemployment."""
    rng = np.random.default_rng(seed)
    years = np.arange(2005, 2025)
    n = len(years)

    # Independent AR-flavored walks, Pakistan-plausible levels
    inflation = 8.0 + np.cumsum(rng.normal(0, 1.8, n))
    inflation = np.clip(inflation, 2.5, 30.0)

    unemployment = 6.0 + np.cumsum(rng.normal(0, 0.35, n))
    unemployment = np.clip(unemployment, 3.0, 11.0)

    return pd.DataFrame({"Year": years,
                         "Inflation": inflation,
                         "Unemployment": unemployment})


def generate_dual_axis_chart():
    df = build_data()
    r = np.corrcoef(df["Inflation"], df["Unemployment"])[0, 1]

    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(15, 6))

    # ------------------------------------------------------------------
    # PANEL A: dual y-axis, right scale fitted to force visual agreement
    # ------------------------------------------------------------------
    slope, intercept = np.polyfit(df["Unemployment"], df["Inflation"], 1)

    ax_a.plot(df["Year"], df["Inflation"], color=C_INFL, linewidth=2.2,
              marker="o", markersize=5, label="Inflation (left axis)")
    ax_a.set_ylabel("Inflation Rate (%)", color=C_INFL, fontsize=11, labelpad=10)
    ax_a.tick_params(axis="y", labelcolor=C_INFL)

    ax_a2 = ax_a.twinx()
    ax_a2.plot(df["Year"], df["Unemployment"], color=C_UNEMP, linewidth=2.2,
               marker="s", markersize=5, label="Unemployment (right axis)")
    ax_a2.set_ylabel("Unemployment Rate (%)", color=C_UNEMP, fontsize=11, labelpad=10)
    ax_a2.tick_params(axis="y", labelcolor=C_UNEMP)

    # The manipulation: choose the right-axis range so the fitted line
    # of unemployment lands on top of the inflation series.
    lo_a, hi_a = ax_a.get_ylim()
    ax_a2.set_ylim((lo_a - intercept) / slope, (hi_a - intercept) / slope)

    ax_a.set_title("A. Dual Y-Axis: Manufactured Co-Movement",
                   fontsize=13, fontweight="bold", pad=14)
    ax_a.set_xlabel("Year", fontsize=11, labelpad=10)
    ax_a.annotate("Right axis rescaled until the lines agree.\n"
                  f"Underlying correlation is only r = {r:.2f}.",
                  xy=(0.97, 0.05), xycoords="axes fraction", ha="right",
                  fontsize=10, color="#333333",
                  bbox=dict(boxstyle="round,pad=0.45", facecolor="#FFF3CD",
                            edgecolor="#D9A404", linewidth=1))

    lines = ax_a.get_lines() + ax_a2.get_lines()
    ax_a.legend(lines, [l.get_label() for l in lines],
                loc="upper left", frameon=True, fontsize=10)

    # ------------------------------------------------------------------
    # PANEL B: percent change from a unified base, one common scale
    # ------------------------------------------------------------------
    base_infl = df["Inflation"].iloc[0]
    base_unemp = df["Unemployment"].iloc[0]
    pct_infl = (df["Inflation"] / base_infl - 1) * 100
    pct_unemp = (df["Unemployment"] / base_unemp - 1) * 100

    ax_b.plot(df["Year"], pct_infl, color=C_INFL, linewidth=2.2,
              marker="o", markersize=5, label="Inflation")
    ax_b.plot(df["Year"], pct_unemp, color=C_UNEMP, linewidth=2.2,
              marker="s", markersize=5, label="Unemployment")
    ax_b.axhline(0, color="#333333", linewidth=1.2)

    ax_b.text(df["Year"].iloc[-1] + 0.3, pct_infl.iloc[-1],
              f"{pct_infl.iloc[-1]:+.0f}%", color=C_INFL,
              va="center", fontsize=10, fontweight="bold")
    ax_b.text(df["Year"].iloc[-1] + 0.3, pct_unemp.iloc[-1],
              f"{pct_unemp.iloc[-1]:+.0f}%", color=C_UNEMP,
              va="center", fontsize=10, fontweight="bold")

    ax_b.set_title("B. Common Scale: Percent Change from 2005 Base",
                   fontsize=13, fontweight="bold", pad=14)
    ax_b.set_xlabel("Year", fontsize=11, labelpad=10)
    ax_b.set_ylabel("Change Since 2005 (%)", fontsize=11, labelpad=10)
    ax_b.set_xlim(2004.5, 2026.5)
    ax_b.legend(loc="upper left", frameon=True, fontsize=10)
    ax_b.annotate("One axis, one baseline.\nInflation rose roughly three times as much.",
                  xy=(0.97, 0.05), xycoords="axes fraction", ha="right",
                  fontsize=10, color="#333333",
                  bbox=dict(boxstyle="round,pad=0.45", facecolor="#E3F2E1",
                            edgecolor="#4C8C4A", linewidth=1))

    ticks = list(range(2005, 2025, 3))
    for ax in (ax_a, ax_b):
        ax.set_xticks(ticks)
        ax.set_xticklabels([str(t) for t in ticks])
        ax.spines["top"].set_visible(False)
        ax.grid(True, alpha=0.3, linewidth=0.6)
        ax.set_axisbelow(True)
    ax_b.spines["right"].set_visible(False)

    fig.suptitle("The Same Two Series, Twice: Dual Axes Invent a Relationship "
                 "That a Common Scale Immediately Refutes",
                 fontsize=15, fontweight="bold", y=1.0)

    plt.tight_layout()

    svg_path = os.path.join(OUTPUT_DIR, "Fig_01_dual_axis.svg")
    fig.savefig(svg_path, format="svg", bbox_inches="tight", dpi=300)
    fig.savefig(os.path.join(OUTPUT_DIR, "Fig_01_dual_axis.png"),
                format="png", bbox_inches="tight", dpi=200)
    print(f"Successfully generated: {svg_path}  (r = {r:.3f})")
    plt.close()


if __name__ == "__main__":
    generate_dual_axis_chart()

Successfully generated: Figures/Fig_01_dual_axis.svg  (r = 0.505)
